Linear Regression Model Training

In [32]:


from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, Side



BASE_DIR = Path.cwd()
if not (BASE_DIR / "model_training").exists() and BASE_DIR.name == "model_training":
    
    BASE_DIR = BASE_DIR.parent

CSV_PATH = BASE_DIR / "model_training" / "dataset_with_model_first_name.csv"
TEMPLATE_PATH = BASE_DIR / "model_training" / "Template_for_Model.xlsx"
OUTPUT_PATH = BASE_DIR / "model_training" / "Template_for_Model_Filled.xlsx"

MONTHS = ["2025-11", "2025-12", "2026-01", "2026-02"]
DATA_START_ROW = 4  


def _first_existing_column(df, candidates):
    """Return the first matching column name from candidates, else None."""
    return next((col for col in candidates if col in df.columns), None)


def _create_fallback_template(template_path):
    """Create a minimal template if the expected file is missing."""
    wb = Workbook()
    ws = wb.active
    ws.title = "Vehicle Prices"
    ws["A1"] = "Vehicle Price Forecast Template (Auto-generated)"

    headers = [
        "Make",
        "Model",
        "Year of Manufacture",
        "NOV 2025",
        "DEC 2025",
        "JAN 2026",
        "FEB 2026",
        "MARCH 2026",
        "APRIL 2026",
        "Next Week Price",
        "AVG. Price | AVG. Mileage",
    ]
    for idx, title in enumerate(headers, start=1):
        cell = ws.cell(row=3, column=idx, value=title)
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center")

    wb.save(template_path)
    print(f"Template not found. Created fallback template -> {template_path}")



def load_and_prepare(csv_path):
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    df = pd.read_csv(csv_path)

    required = ["Make", "Model", "Year", "Price", "published date"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required column(s): {missing}")

    df["published date"] = pd.to_datetime(df["published date"], errors="coerce")
    df = df.dropna(subset=["published date"])
    df["month"] = df["published date"].dt.to_period("M").astype(str)
    return df


def build_monthly_pivot(df, months):
    """Pivot: rows = (Make, Model, Year), columns = month, values = avg Price."""
    pivot = (
        df.groupby(["Make", "Model", "Year", "month"])["Price"]
        .mean()
        .unstack("month")
    )
    pivot.columns = [str(c) for c in pivot.columns]
    for m in months:
        if m not in pivot.columns:
            pivot[m] = np.nan
    return pivot[months].reset_index()


def compute_global_month_avg(df, months):
    """Returns dict: {month_str: global_avg_price} with safe fallback."""
    overall_avg = df["Price"].mean()
    monthly = {m: df.loc[df["month"] == m, "Price"].mean() for m in months}
    return {m: (monthly[m] if not pd.isna(monthly[m]) else overall_avg) for m in months}


def predict_future_months(nov, dec, jan, feb, global_avg_jan):
    """Predict MARCH (step 5) and APRIL (step 6) using linear trend."""
    known = [(1, nov), (2, dec), (3, jan), (4, feb)]
    known_vals = [
        (x, y) for x, y in known if y is not None and not (isinstance(y, float) and np.isnan(y))
    ]

    if len(known_vals) >= 2:
        xs = np.array([v[0] for v in known_vals])
        ys = np.array([v[1] for v in known_vals])
        coeffs = np.polyfit(xs, ys, 1)  
        march = round(float(np.polyval(coeffs, 5)))
        april = round(float(np.polyval(coeffs, 6)))
    elif len(known_vals) == 1:
        # Only one data point: hold flat.
        march = round(known_vals[0][1])
        april = round(known_vals[0][1])
    else:
        # No actual data: use global trend proxy.
        march = round(global_avg_jan * 0.97)
        april = round(global_avg_jan * 0.95)

    return march, april



def cell_val(value, is_predicted):
    """Return numeric for actual, 'n#prd' formatted string for predicted."""
    if is_predicted:
        return f"{int(round(value)):,}#prd"
    return int(round(value))



def fill_template(final_df, global_month_avg, template_path, output_path):
    if not template_path.exists():
        _create_fallback_template(template_path)

    wb = load_workbook(template_path)
    ws = wb.active

    # Styles
    actual_font = Font(name="Aptos Narrow", size=11)
    pred_font = Font(name="Aptos Narrow", size=11, color="0070C0")  # blue
    thin = Side(style="thin")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for i, row_data in final_df.iterrows():
        excel_row = DATA_START_ROW + i

        nov_raw = row_data.get("2025-11")
        dec_raw = row_data.get("2025-12")
        jan_raw = row_data.get("2026-01")
        feb_raw = row_data.get("2026-02")

        nov_actual = None if pd.isna(nov_raw) else round(nov_raw)
        dec_actual = None if pd.isna(dec_raw) else round(dec_raw)
        jan_actual = None if pd.isna(jan_raw) else round(jan_raw)
        feb_actual = None if pd.isna(feb_raw) else round(feb_raw)

        nov_is_pred = nov_actual is None
        dec_is_pred = dec_actual is None
        jan_is_pred = jan_actual is None
        feb_is_pred = feb_actual is None

        nov_val = nov_actual if not nov_is_pred else global_month_avg["2025-11"]
        dec_val = dec_actual if not dec_is_pred else global_month_avg["2025-12"]
        jan_val = jan_actual if not jan_is_pred else global_month_avg["2026-01"]
        feb_val = feb_actual if not feb_is_pred else global_month_avg["2026-02"]

        march_pred, april_pred = predict_future_months(
            nov_actual,
            dec_actual,
            jan_actual,
            feb_actual,
            global_avg_jan=global_month_avg["2026-01"],
        )

        year_val = int(row_data["Year"]) if not pd.isna(row_data["Year"]) else None
        avg_mileage_v = None if pd.isna(row_data["AvgMileage"]) else round(row_data["AvgMileage"])
        avg_price_v = None if pd.isna(row_data["AvgPrice"]) else round(row_data["AvgPrice"])
        avg_combined = (
            f"{avg_price_v:,} | {avg_mileage_v:,}"
            if avg_price_v is not None and avg_mileage_v is not None
            else None
        )

        row_values = [
            (row_data["Make"], False, None),  # A
            (row_data["Model"], False, None),  # B
            (year_val, False, "center"),  # C
            (cell_val(nov_val, nov_is_pred), nov_is_pred, None),  # D
            (cell_val(dec_val, dec_is_pred), dec_is_pred, None),  # E
            (cell_val(jan_val, jan_is_pred), jan_is_pred, None),  # F
            (cell_val(feb_val, feb_is_pred), feb_is_pred, None),  # G
            (f"{march_pred:,}#prd", True, None),  # H
            (f"{april_pred:,}#prd", True, None),  # I
            (f"{april_pred:,}#prd", True, None),  # J
            (avg_combined, False, None),  # K
        ]

        for col_idx, (val, is_pred, align) in enumerate(row_values, start=1):
            cell = ws.cell(row=excel_row, column=col_idx, value=val)
            cell.font = pred_font if is_pred else actual_font
            cell.border = border
            if align:
                cell.alignment = Alignment(horizontal=align)
            if col_idx in (4, 5, 6, 7) and not is_pred:
                cell.number_format = "#,##0"

    col_widths = {
        "A": 18,
        "B": 22,
        "C": 10,
        "D": 16,
        "E": 16,
        "F": 16,
        "G": 16,
        "H": 18,
        "I": 28,
        "J": 18,
        "K": 24,
    }
    for col_letter, width in col_widths.items():
        ws.column_dimensions[col_letter].width = width

    wb.save(output_path)
    print(f"Saved -> {output_path}")


# -----------------------------
# MAIN
# -----------------------------
def main():
    print("Loading data...")
    df = load_and_prepare(CSV_PATH)

    print("Building monthly price pivot...")
    pivot = build_monthly_pivot(df, MONTHS)

    print("Computing global monthly averages...")
    global_month_avg = compute_global_month_avg(df, MONTHS)
    for m, avg in global_month_avg.items():
        print(f"  {m}: {avg:,.0f}")

    print("Aggregating avg mileage and overall avg price per vehicle...")
    mileage_col = _first_existing_column(df, ["Milleage", "Mileage"])
    if mileage_col is None:
        raise KeyError("Neither 'Milleage' nor 'Mileage' column exists in the dataset.")

    avg_mileage = (
        df.groupby(["Make", "Model", "Year"])[mileage_col]
        .mean()
        .reset_index()
        .rename(columns={mileage_col: "AvgMileage"})
    )
    avg_price_overall = (
        df.groupby(["Make", "Model", "Year"])["Price"]
        .mean()
        .reset_index()
        .rename(columns={"Price": "AvgPrice"})
    )

    final = pivot.merge(avg_mileage, on=["Make", "Model", "Year"], how="left")
    final = final.merge(avg_price_overall, on=["Make", "Model", "Year"], how="left")
    print(f"Total unique vehicles: {len(final)}")

    print("Writing to Excel template...")
    fill_template(final, global_month_avg, TEMPLATE_PATH, OUTPUT_PATH)
    print(f"\nDone! {len(final)} vehicles written to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Loading data...
Building monthly price pivot...
Computing global monthly averages...
  2025-11: 7,695,691
  2025-12: 7,422,176
  2026-01: 7,114,552
  2026-02: 6,875,422
Aggregating avg mileage and overall avg price per vehicle...
Total unique vehicles: 4438
Writing to Excel template...
Saved -> c:\Users\Pc\Documents\GitHub\autoinsightcs70\model_training\Template_for_Model_Filled.xlsx

Done! 4438 vehicles written to c:\Users\Pc\Documents\GitHub\autoinsightcs70\model_training\Template_for_Model_Filled.xlsx


Gradient Boosting Model Training

In [33]:
"""
Vehicle Price Forecasting — LightGBM GradientBoosting Regressor
================================================================
Uses LightGBM (preferred) or sklearn GradientBoostingRegressor (fallback)
to predict monthly vehicle prices.  Output Excel structure is unchanged.
"""

from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, Side
from sklearn.metrics import mean_absolute_error

# ------------------------------------------------------------------
# Try LightGBM first; fall back to sklearn GradientBoostingRegressor
# ------------------------------------------------------------------
try:
    import lightgbm as lgb
    _USE_LGBM = True
    print("[Model] Using LightGBM GradientBoosting Regressor")
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor as _SKGBR
    _USE_LGBM = False
    print("[Model] LightGBM not found — using sklearn GradientBoostingRegressor")

# -----------------------------
# CONFIG
# -----------------------------
BASE_DIR = Path.cwd()
if not (BASE_DIR / "model_training").exists() and BASE_DIR.name == "model_training":
    BASE_DIR = BASE_DIR.parent

CSV_PATH      = BASE_DIR / "model_training" / "dataset_with_condition.csv"
TEMPLATE_PATH = BASE_DIR / "model_training" / "Template_for_Model.xlsx"
OUTPUT_PATH   = BASE_DIR / "model_training" / "Template_for_Model_Filled.xlsx"

MONTHS = ["2025-11", "2025-12", "2026-01", "2026-02"]
PREDICT_MONTHS = ["2026-03", "2026-04"]   # March & April
DATA_START_ROW = 4


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def _first_existing_column(df, candidates):
    return next((c for c in candidates if c in df.columns), None)


def _create_fallback_template(template_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Vehicle Prices"
    ws["A1"] = "Vehicle Price Forecast Template (Auto-generated)"
    headers = [
        "Make", "Model", "Year of Manufacture",
        "NOV 2025", "DEC 2025", "JAN 2026", "FEB 2026",
        "MARCH 2026", "APRIL 2026", "Next Week Price",
        "AVG. Price | AVG. Mileage",
    ]
    for idx, title in enumerate(headers, start=1):
        cell = ws.cell(row=3, column=idx, value=title)
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center")
    wb.save(template_path)
    print(f"Template not found — created fallback: {template_path}")


# ------------------------------------------------------------------
# Data loading
# ------------------------------------------------------------------
def load_and_prepare(csv_path):
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    required = ["Make", "Model", "Year", "Price", "published date"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

    df["published date"] = pd.to_datetime(df["published date"], errors="coerce")
    df = df.dropna(subset=["published date"])
    df["month"] = df["published date"].dt.to_period("M").astype(str)
    df["month_num"] = df["published date"].dt.year * 12 + df["published date"].dt.month
    return df


# ------------------------------------------------------------------
# Monthly pivot (actuals)
# ------------------------------------------------------------------
def build_monthly_pivot(df, months):
    pivot = (
        df.groupby(["Make", "Model", "Year", "month"])["Price"]
        .mean()
        .unstack("month")
    )
    pivot.columns = [str(c) for c in pivot.columns]
    for m in months:
        if m not in pivot.columns:
            pivot[m] = np.nan
    return pivot[months].reset_index()


def compute_global_month_avg(df, months):
    overall = df["Price"].mean()
    return {
        m: df.loc[df["month"] == m, "Price"].mean() if (df["month"] == m).any() else overall
        for m in months
    }


# ------------------------------------------------------------------
# Feature engineering for the ML model
# ------------------------------------------------------------------
def build_feature_matrix(df):
    """
    Build (X, y) from the raw listing DataFrame.
    Features: Make (encoded), Model (encoded), Year, month_num,
              log_mileage (if available), condition dummies (if available).
    Target:   Price
    """
    feat = df[["Make", "Model", "Year", "month_num", "Price"]].copy()

    # Mileage
    mileage_col = _first_existing_column(df, ["Milleage", "Mileage"])
    if mileage_col:
        feat["log_mileage"] = np.log1p(df[mileage_col].fillna(df[mileage_col].median()))
    else:
        feat["log_mileage"] = 0.0

    # Condition (one-hot)
    if "condition" in df.columns:
        dummies = pd.get_dummies(df["condition"].fillna("Unknown"), prefix="cond")
        feat = pd.concat([feat, dummies.reset_index(drop=True)], axis=1)

    # Label-encode Make & Model
    feat["make_enc"]  = feat["Make"].astype("category").cat.codes
    feat["model_enc"] = feat["Model"].astype("category").cat.codes

    # Store encoders (category → code) for inference later
    make_map  = dict(enumerate(feat["Make"].astype("category").cat.categories))
    model_map = dict(enumerate(feat["Model"].astype("category").cat.categories))
    make_enc_map  = {v: k for k, v in make_map.items()}
    model_enc_map = {v: k for k, v in model_map.items()}

    drop_cols = ["Make", "Model", "Price"]
    X = feat.drop(columns=drop_cols)
    y = feat["Price"]

    cond_cols = [c for c in feat.columns if c.startswith("cond_")]

    return X, y, make_enc_map, model_enc_map, cond_cols


# ------------------------------------------------------------------
# Model training
# ------------------------------------------------------------------
def train_model(X, y):
    """Train LightGBM or sklearn GBR and return fitted model."""
    if _USE_LGBM:
        model = lgb.LGBMRegressor(
            n_estimators=600,
            learning_rate=0.05,
            num_leaves=63,
            max_depth=-1,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            random_state=42,
            n_jobs=-1,
        )
    else:
        model = _SKGBR(
            n_estimators=600,
            learning_rate=0.05,
            max_depth=5,
            min_samples_leaf=20,
            subsample=0.8,
            max_features=0.8,
            random_state=42,
        )

    model.fit(X, y)
    return model


# ------------------------------------------------------------------
# Prediction helpers
# ------------------------------------------------------------------
def _month_str_to_num(month_str):
    """'2026-03' → 2026*12+3 = 24315"""
    y, m = map(int, month_str.split("-"))
    return y * 12 + m


def predict_row(model, make, model_name, year,
                make_enc_map, model_enc_map, cond_cols,
                month_str, default_log_mileage):
    """Build a single-row feature vector and return model prediction."""
    month_num = _month_str_to_num(month_str)
    make_enc  = make_enc_map.get(make, -1)
    model_enc = model_enc_map.get(model_name, -1)

    row = {
        "Year":           year,
        "month_num":      month_num,
        "log_mileage":    default_log_mileage,
        "make_enc":       make_enc,
        "model_enc":      model_enc,
    }
    for c in cond_cols:
        row[c] = 0  # neutral / unknown condition

    feat_row = pd.DataFrame([row])
    pred = model.predict(feat_row)[0]
    return max(0, round(float(pred)))


# ------------------------------------------------------------------
# Cell value formatter (unchanged from original)
# ------------------------------------------------------------------
def cell_val(value, is_predicted):
    if is_predicted:
        return f"{int(round(value)):,}#prd"
    return int(round(value))


# ------------------------------------------------------------------
# Excel writer (structure unchanged from original)
# ------------------------------------------------------------------
def fill_template(final_df, global_month_avg,
                  model, make_enc_map, model_enc_map, cond_cols,
                  template_path, output_path):

    if not template_path.exists():
        _create_fallback_template(template_path)

    wb = load_workbook(template_path)
    ws = wb.active

    actual_font = Font(name="Aptos Narrow", size=11)
    pred_font   = Font(name="Aptos Narrow", size=11, color="0070C0")
    thin   = Side(style="thin")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for i, row_data in final_df.iterrows():
        excel_row = DATA_START_ROW + i

        nov_raw = row_data.get("2025-11")
        dec_raw = row_data.get("2025-12")
        jan_raw = row_data.get("2026-01")
        feb_raw = row_data.get("2026-02")

        nov_actual = None if pd.isna(nov_raw) else round(nov_raw)
        dec_actual = None if pd.isna(dec_raw) else round(dec_raw)
        jan_actual = None if pd.isna(jan_raw) else round(jan_raw)
        feb_actual = None if pd.isna(feb_raw) else round(feb_raw)

        nov_is_pred = nov_actual is None
        dec_is_pred = dec_actual is None
        jan_is_pred = jan_actual is None
        feb_is_pred = feb_actual is None

        nov_val = nov_actual if not nov_is_pred else round(global_month_avg["2025-11"])
        dec_val = dec_actual if not dec_is_pred else round(global_month_avg["2025-12"])
        jan_val = jan_actual if not jan_is_pred else round(global_month_avg["2026-01"])
        feb_val = feb_actual if not feb_is_pred else round(global_month_avg["2026-02"])

        # ---- LightGBM / GBR predictions for March & April --------
        make_name   = row_data["Make"]
        model_name  = row_data["Model"]
        year        = int(row_data["Year"]) if not pd.isna(row_data["Year"]) else 2020

        # Use AvgMileage for the log-mileage feature
        avg_mileage_v = row_data.get("AvgMileage")
        default_log_mileage = (
            np.log1p(avg_mileage_v)
            if avg_mileage_v is not None and not pd.isna(avg_mileage_v)
            else 0.0
        )

        march_pred = predict_row(
            model, make_name, model_name, year,
            make_enc_map, model_enc_map, cond_cols,
            "2026-03", default_log_mileage,
        )
        april_pred = predict_row(
            model, make_name, model_name, year,
            make_enc_map, model_enc_map, cond_cols,
            "2026-04", default_log_mileage,
        )
        # -----------------------------------------------------------

        year_val       = int(row_data["Year"]) if not pd.isna(row_data["Year"]) else None
        avg_mileage_v2 = None if pd.isna(row_data["AvgMileage"]) else round(row_data["AvgMileage"])
        avg_price_v    = None if pd.isna(row_data["AvgPrice"])   else round(row_data["AvgPrice"])
        avg_combined   = (
            f"{avg_price_v:,} | {avg_mileage_v2:,}"
            if avg_price_v is not None and avg_mileage_v2 is not None
            else None
        )

        row_values = [
            (row_data["Make"],                       False,    None),     # A
            (row_data["Model"],                      False,    None),     # B
            (year_val,                               False,    "center"), # C
            (cell_val(nov_val, nov_is_pred),         nov_is_pred, None),  # D
            (cell_val(dec_val, dec_is_pred),         dec_is_pred, None),  # E
            (cell_val(jan_val, jan_is_pred),         jan_is_pred, None),  # F
            (cell_val(feb_val, feb_is_pred),         feb_is_pred, None),  # G
            (f"{march_pred:,}#prd",                  True,    None),      # H
            (f"{april_pred:,}#prd",                  True,    None),      # I
            (f"{april_pred:,}#prd",                  True,    None),      # J
            (avg_combined,                           False,    None),     # K
        ]

        for col_idx, (val, is_pred, align) in enumerate(row_values, start=1):
            cell = ws.cell(row=excel_row, column=col_idx, value=val)
            cell.font   = pred_font if is_pred else actual_font
            cell.border = border
            if align:
                cell.alignment = Alignment(horizontal=align)
            if col_idx in (4, 5, 6, 7) and not is_pred:
                cell.number_format = "#,##0"

    col_widths = {
        "A": 18, "B": 22, "C": 10,
        "D": 16, "E": 16, "F": 16, "G": 16,
        "H": 18, "I": 28, "J": 18, "K": 24,
    }
    for col_letter, width in col_widths.items():
        ws.column_dimensions[col_letter].width = width

    wb.save(output_path)
    print(f"Saved → {output_path}")


# ------------------------------------------------------------------
# MAIN
# ------------------------------------------------------------------
def main():
    print("Loading data …")
    df = load_and_prepare(CSV_PATH)

    print("Training GradientBoosting model …")
    X, y, make_enc_map, model_enc_map, cond_cols = build_feature_matrix(df)
    model = train_model(X, y)

    # Quick in-sample score (R²)
    from sklearn.metrics import r2_score, mean_absolute_error
    y_pred_train = model.predict(X)
    r2  = r2_score(y, y_pred_train)
    

    mae = mean_absolute_error(y, y_pred_train)
    print(f"  Train R²  : {r2:.4f}")
    print(f"  Train MAE : {mae:,.0f}")
    

    print("Building monthly pivot …")
    pivot = build_monthly_pivot(df, MONTHS)

    print("Computing global monthly averages …")
    global_month_avg = compute_global_month_avg(df, MONTHS)
    for m, avg in global_month_avg.items():
        print(f"  {m}: {avg:,.0f}")

    print("Aggregating avg mileage and avg price …")
    mileage_col = _first_existing_column(df, ["Milleage", "Mileage"])
    if mileage_col is None:
        raise KeyError("Neither 'Milleage' nor 'Mileage' column found.")

    avg_mileage = (
        df.groupby(["Make", "Model", "Year"])[mileage_col]
        .mean().reset_index().rename(columns={mileage_col: "AvgMileage"})
    )
    avg_price_overall = (
        df.groupby(["Make", "Model", "Year"])["Price"]
        .mean().reset_index().rename(columns={"Price": "AvgPrice"})
    )

    final = pivot.merge(avg_mileage,      on=["Make", "Model", "Year"], how="left")
    final = final.merge(avg_price_overall, on=["Make", "Model", "Year"], how="left")
    print(f"Total unique vehicles: {len(final)}")

    print("Writing to Excel template …")
    fill_template(
        final, global_month_avg,
        model, make_enc_map, model_enc_map, cond_cols,
        TEMPLATE_PATH, OUTPUT_PATH,
    )
    print(f"\nDone! {len(final)} vehicles written to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

[Model] Using LightGBM GradientBoosting Regressor
Loading data …
Training GradientBoosting model …
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000807 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 646
[LightGBM] [Info] Number of data points in the train set: 23430, number of used features: 5
[LightGBM] [Info] Start training from score 7196434.305719
  Train R²  : 0.9078
  Train MAE : 797,595
Building monthly pivot …
Computing global monthly averages …
  2025-11: 7,695,691
  2025-12: 7,422,176
  2026-01: 7,114,552
  2026-02: 6,875,422
Aggregating avg mileage and avg price …
Total unique vehicles: 9904
Writing to Excel template …
Saved → c:\Users\Pc\Documents\GitHub\autoinsightcs70\model_training\Template_for_Model_Filled.xlsx

Done! 9904 vehicles written to c:\Users\Pc\Documents\GitHub\autoinsightcs70\model_training\Template_for_Model_Filled.xlsx


In [34]:
def main():
    print("Loading data …")
    df = load_and_prepare(CSV_PATH)

    print("Training GradientBoosting model …")
    X, y, make_enc_map, model_enc_map, cond_cols = build_feature_matrix(df)
    model = train_model(X, y)

    # ============================================================
    # CALCULATE REGRESSION METRICS ONLY
    # ============================================================
    from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
    
    y_pred_train = model.predict(X)
    
    r2 = r2_score(y, y_pred_train)
    mae = mean_absolute_error(y, y_pred_train)
    mse = mean_squared_error(y, y_pred_train)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y, y_pred_train)
    
    # Print Results
    print("\n" + "="*60)
    print("MODEL EVALUATION METRICS")
    print("="*60)
    print(f"  R² Score (Variance Explained)  : {r2:.4f}")
    print(f"  MAE (Avg Error in Currency)    : {mae:,.0f}")
    print(f"  RMSE (Root Mean Squared Error) : {rmse:,.0f}")
    print(f"  MAPE (Error as %)              : {mape:.2%}")
    print("="*60 + "\n")


if __name__ == "__main__":
    main()

Loading data …
Training GradientBoosting model …
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000711 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 646
[LightGBM] [Info] Number of data points in the train set: 23430, number of used features: 5
[LightGBM] [Info] Start training from score 7196434.305719

MODEL EVALUATION METRICS
  R² Score (Variance Explained)  : 0.9078
  MAE (Avg Error in Currency)    : 797,595
  RMSE (Root Mean Squared Error) : 1,527,857
  MAPE (Error as %)              : 36.29%

